In [47]:
#install Dependecies 
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [48]:
# Load env Variables
from dotenv import load_dotenv
load_dotenv()

True

In [49]:
#Create an API client 
from anthropic import Anthropic
client = Anthropic()
model = "claude-sonnet-4-5"# video instruct you to use claude code 4.0 which is depricate content is dated 
#we will acess claude using create fucntion 
#this create function requires a model parameter max tokens and message 
# claude does not try to target max tokens it acts more as a guardrail 
# messages are important 
#user message is written by person 

# 

In [50]:
#helper functions to maintain user history 
def add_user_message(messages,text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages,text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)
    
def chat(messages): #message history 
    message = client.messages.create(
        model= model,
        max_tokens=1000,
        messages= messages,
    )
    return message.content[0].text

#Make a Reuquest
message = client.messages.create(
    model= model,
    max_tokens=1000,
    messages=[
        #create a dictionary with role of user and content
        {
            "role": "user", 
            "content": "What is quantum computing? Answer in one sentence."
            }
    ]
)
# example to show that claude has no memory is these sorts of blocks to
#message = client.messages.create(
    #model= model,
    #max_tokens=1000,
    #messages=[
        #create a dictionary with role of user and content
       # {
           # "role": "user", 
            #"content": "write another sentence."
           # }
    #]
#)

In [51]:
# make a starting list of messages 
# add in the inital user quesion of define quantum computing
messages = []

add_user_message(messages, "What is quantum computing? Answer in one sentence.")
messages
# pass the list of messages into chat to get an answer
answer = chat(messages)
# take the answer and add it as an assistant message into our list 
add_assistant_message(messages, answer)
messages 
# add in the user's follow up questions 
add_user_message(messages, "write another sentence.")
answer = chat(messages)
answer

'Quantum computers use quantum bits (qubits) that can exist in multiple states simultaneously, unlike classical bits that can only be either 0 or 1.'

In [52]:
#extracting the Repsonse 
#message.content[0].text
#message.content[0].text# we see here that the  next sentence is random thus no memory from the first message request 

In [53]:
#claude doesnt store any of your conversation history 
# this means if you ever want a transcript of your conversation, you'll need to save it yourself.

In [54]:
#make an initial list of messages 
#messages = []
#use a while true loop to run the chat bot indefintley 
#while True:
    #get user input 
 #   user_input = input("> ")
    #print(">", user_input)
    
    #add user input to the list of messages 
  #  add_user_message(messages, user_input)
    
    #call claude with the chat union 
   # answer = chat(messages)
    # add generated  text to list of messages 
    #add_assistant_message(messages, answer)
    #print generat4d text 
    #print("Assistant:", answer)
    #print("---")
    # simple implementation of a looping chat bot 

In [55]:
# sysstem prompt example 
#system_prompt ="You are a patient math tut"
messages = []

add_user_message(messages, "how do i solve 5 + 3x = 43 for x?")
answer = chat(messages)


answer

'# Solving 5 + 3x = 43\n\nTo solve for x, isolate it by following these steps:\n\n**Step 1:** Subtract 5 from both sides\n```\n5 + 3x = 43\n3x = 43 - 5\n3x = 38\n```\n\n**Step 2:** Divide both sides by 3\n```\nx = 38/3\n```\n\n**Answer:** x = 38/3 (or 12⅔, or approximately 12.67)\n\n---\n\n**Check your work:**\n- 5 + 3(38/3) = 5 + 38 = 43 ✓'

In [56]:
#When building a chat app with claude there is a signifcant user experience challenge. Repsonse can take 10-30 secnds to generae, leaving users stting at a loading spinner. the solution is repsonse streaming 
# make a list of messages and manually call 
messages = []

add_user_message(messages, "Write a 1 sentence desciption of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)
for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_011Cef9mSPReyZgqgNrmfuCE', container=None, content=[], model='claude-sonnet-4-5-20250929', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=20, output_tokens=1, output_tokens_details=None, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='A', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' fake database is a simulated or mock data storage system that mimics the structure and behavior of a real database but contains fabricated, sample, or test', type='tex

In [57]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream(
model = model,
max_tokens=1000,
messages= messages 
)as stream:
    for text in stream.text_stream:
       # print(text, end ="")
       pass 
stream.get_final_message()

ParsedMessage(id='msg_011Cef9mawNq8V3vT5wAzXHK', container=None, content=[ParsedTextBlock(citations=None, text='A relational database containing fabricated customer records, synthetic transaction histories, and randomly generated employee data used for software testing and developer training purposes.', type='text', parsed_output=None)], model='claude-sonnet-4-5-20250929', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=18, output_tokens=31, output_tokens_details=None, server_tool_use=None, service_tier='standard'))

In [58]:
messages = []
add_user_message(messages, "Count from 1 to 10.")

chat(messages, stop_sequences=["5", "3,4"])

TypeError: chat() got an unexpected keyword argument 'stop_sequences'

In [59]:
import json

clean_json = json.loads(text.strip())

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
messages = []
add_user_message(messages, "Generate a very short event bridge rule as JSON")
#assistance message
add_assistant_message(messages, "```json")

chat(messages, stop_sequences=["```"])


TypeError: chat() got an unexpected keyword argument 'stop_sequences'